# Fact Color Usage — Gold Layer

Aggregated fact table at the grain of one row per color. Answers how widely each color is used across sets, parts, themes, and categories.

## What this notebook does

1. **Read** — Loads `fct_set_inventory` and `dim_color` from gold.
2. **Transform** — Aggregates per color: total sets, total parts, total quantity, distinct part categories, and the top theme using the color.
3. **Write** — Overwrites the Delta table at `/Volumes/lego_brickbase/gold/delta_volume/fct_color_usage`.
4. **Register** — Creates the catalog table `lego_brickbase.gold.fct_color_usage`.

## Imports and Configuration

In [ ]:
from pyspark.sql import SparkSession, Window
from pyspark.sql.functions import (
    col, countDistinct, sum as _sum, row_number,
)


GOLD_FCT_SET_INVENTORY = "lego_brickbase.gold.fct_set_inventory"
GOLD_DIM_COLOR         = "lego_brickbase.gold.dim_color"
DELTA_TABLE_PATH       = "/Volumes/lego_brickbase/gold/delta_volume/fct_color_usage"
CATALOG_TABLE          = "lego_brickbase.gold.fct_color_usage"

## Read and Transform

Aggregate `fct_set_inventory` to the color grain and enrich with color dimension attributes.

In [ ]:
df_inv = spark.table(GOLD_FCT_SET_INVENTORY)
df_color = spark.table(GOLD_DIM_COLOR)

# Core metrics per color
df_color_agg = (
    df_inv
    .groupBy("color_key")
    .agg(
        countDistinct("set_key").alias("total_sets_featuring_color"),
        countDistinct("part_key").alias("total_parts_in_color"),
        _sum("quantity").alias("total_quantity"),
        countDistinct("part_category_name").alias("distinct_part_categories"),
    )
)

# Top theme per color (theme with highest total quantity of this color)
w = Window.partitionBy("color_key").orderBy(col("theme_qty").desc())
df_top_theme = (
    df_inv
    .groupBy("color_key", "root_theme_name")
    .agg(_sum("quantity").alias("theme_qty"))
    .withColumn("rn", row_number().over(w))
    .filter(col("rn") == 1)
    .select(
        col("color_key").alias("_tt_color_key"),
        col("root_theme_name").alias("top_theme"),
    )
)

# Join with dim_color and top theme
df_fct = (
    df_color_agg
    .join(df_color, df_color_agg["color_key"] == df_color["color_key"], "left")
    .join(df_top_theme, df_color_agg["color_key"] == df_top_theme["_tt_color_key"], "left")
    .select(
        df_color_agg["color_key"],
        df_color["color_name"],
        df_color["rgb"],
        df_color["color_family"],
        df_color["is_transperent"],
        col("total_sets_featuring_color"),
        col("total_parts_in_color"),
        col("total_quantity"),
        col("distinct_part_categories"),
        col("top_theme"),
    )
)

df_fct.printSchema()
df_fct.display(10, truncate=False)

## Write to Delta Volume, Register Catalog Table, and Apply Constraints

In [ ]:
(
    df_fct
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(DELTA_TABLE_PATH)
)

row_count = spark.read.format("delta").load(DELTA_TABLE_PATH).count()
print(f"Rows written to Delta table: {row_count:,}")

spark.sql(f"""
    CREATE OR REPLACE TABLE {CATALOG_TABLE} (
        color_key                  INTEGER NOT NULL COMMENT 'Foreign key to dim_color; surrogate key identifying the color',
        color_name                 STRING           COMMENT 'Human-readable name of the color',
        rgb                        STRING           COMMENT 'Six-digit hexadecimal RGB value of the color',
        color_family               STRING           COMMENT 'Derived broad color grouping (e.g. Red, Blue, Green)',
        is_transperent             BOOLEAN          COMMENT 'True if the color is a transparent variant',
        total_sets_featuring_color BIGINT           COMMENT 'Number of distinct sets that include at least one part in this color',
        total_parts_in_color       BIGINT           COMMENT 'Number of distinct part designs available in this color',
        total_quantity             BIGINT           COMMENT 'Total sum of all individual part quantities in this color across all sets',
        distinct_part_categories   BIGINT           COMMENT 'Number of distinct part categories that have parts in this color',
        top_theme                  STRING           COMMENT 'Root theme name with the highest total quantity of parts in this color',
        CONSTRAINT fct_color_usage_pk
            PRIMARY KEY (color_key),
        CONSTRAINT fct_color_usage_fk_color
            FOREIGN KEY (color_key) REFERENCES lego_brickbase.gold.dim_color (color_key)
    )
    COMMENT 'Aggregated fact table at the color grain. Summarises how widely each LEGO color is used across sets, parts, themes, and categories.'
""")

spark.sql(f"""
    INSERT INTO {CATALOG_TABLE}
    SELECT * FROM delta.`{DELTA_TABLE_PATH}`
""")

print(f"Catalog table ready with constraints: {CATALOG_TABLE}")